# cdm_skani CTS Demo

End-to-end demo of the generic (no-refdata) `cdm_skani` CTS tool. Skani is a fast ANI calculator
for genomes / contigs / MAGs; this image lets you compute ANI between user-supplied genomes
without bringing any reference data. For querying against the GTDB R232 representative panel,
use `cdm_skani_gtdb` instead.

- **Image:** `ghcr.io/kbaseincubator/cdm_skani:0.1.0@sha256:3c645fa64e06df1e536a48ef608656ab11c7d5d19fa690c68fee3059effe9da4`
- **Refdata:** none
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/skani_demo/`

See the [cdm_skani repo](https://github.com/kbaseincubator/cdm_skani) for full details.

**What we'll show:**
1. `skani triangle` (all-vs-all ANI across the 4 test genomes; useful for MAG dereplication / clustering)
2. `skani dist` (one query vs one reference; the building block, lower default `--min-af` so we see hits)

Both subcommands work on user-supplied FASTAs only. The 4 test genomes are from 4 different species
(2 bacteria + 2 archaea) and are deliberately distant, so by default skani returns no edges (filtered
out by the 15% aligned-fraction threshold). We narrate that explicitly and re-run `dist` with
`--min-af 0` so the demo also shows a non-empty output.


## 1. Setup

In [1]:
import io, time
import pandas as pd

tscli  = get_task_service_client()
mincli = get_minio_client()

IMAGE      = "ghcr.io/kbaseincubator/cdm_skani:0.1.0@sha256:3c645fa64e06df1e536a48ef608656ab11c7d5d19fa690c68fee3059effe9da4"
OUTPUT_DIR = "cts/io/jplfaria/output/skani_demo"

print(tscli.whoami())


{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Same 4 test assemblies (.fna.gz) the other tool demos use, in `cts/io/gavin/test_files/`.


In [2]:
input_files = sorted(
    f"cts/{o.object_name}"
    for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True)
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna")
)
print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")


4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. skani triangle (all-vs-all)

Submit one container with all 4 inputs. `-E` switches the output from a phylip-format similarity
matrix to a sparse edge list (TSV). Only pairs that clear `--min-af 15` (the default) get a row.

For our 4 deliberately distant test genomes, expect an empty edge list. In real use this subcommand
shines on MAG / isolate datasets where many genomes are >95% ANI to each other.


In [3]:
triangle_job = tscli.submit_job(
    IMAGE,
    input_files,
    f"{OUTPUT_DIR}/triangle",
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "triangle",
        "-E",
        "-o", "/out/triangle.tsv",
        "-t", "4",
        tscli.insert_files(),
    ],
    num_containers=1,
    cpus=4, memory="8GB", runtime="PT15M",
)
print(f"triangle job: {triangle_job.id}")
t0 = time.time()
triangle_job.wait_for_completion()
print(f"  state={triangle_job.get_job_status()['state']}  exit={triangle_job.get_exit_codes().get('exit_codes')}  waited={time.time()-t0:.0f}s")

tri_outs = [o for o in triangle_job.get_job().get("outputs", []) if o["file"].endswith("triangle.tsv")]
bucket, key = tri_outs[0]["file"].split("/", 1)
raw = mincli.get_object(bucket, key).read().decode("utf-8")
print(f"\ntriangle.tsv ({len(raw)} bytes):")
print(raw if raw.strip() else "(header-only, no edges; all 4 test genomes are <15% AF to each other)")


triangle job: 1688e6a5-2aa2-4d06-83ae-8b6e93548b68


  state=complete  exit=[0]  waited=41s

triangle.tsv (84 bytes):
Ref_file	Query_file	ANI	Align_fraction_ref	Align_fraction_query	Ref_name	Query_name



## 4. skani dist (single query vs single reference, with `--min-af 0` for visibility)

Pick the 2 bacterial genomes from the test set (Pelotomaculum and Zinderia). With `--min-af 0`
skani returns a row for any pair, however small the aligned fraction, instead of silently
filtering. This shows the row format the downstream code reads.


In [4]:
bact_inputs = [f for f in input_files if ("000010565" in f or "000147015" in f)]
assert len(bact_inputs) == 2, f"expected 2 bacterial inputs, got {len(bact_inputs)}"
print(f"dist inputs: {bact_inputs}")

dist_job = tscli.submit_job(
    IMAGE,
    bact_inputs,
    f"{OUTPUT_DIR}/dist",
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "dist",
        "--min-af", "0",
        "-o", "/out/dist.tsv",
        "-t", "4",
        tscli.insert_files(),
    ],
    num_containers=1,
    cpus=4, memory="8GB", runtime="PT15M",
)
print(f"dist job: {dist_job.id}")
t0 = time.time()
dist_job.wait_for_completion()
print(f"  state={dist_job.get_job_status()['state']}  exit={dist_job.get_exit_codes().get('exit_codes')}  waited={time.time()-t0:.0f}s")

dist_outs = [o for o in dist_job.get_job().get("outputs", []) if o["file"].endswith("dist.tsv")]
bucket, key = dist_outs[0]["file"].split("/", 1)
raw = mincli.get_object(bucket, key).read().decode("utf-8")
print(f"\ndist.tsv ({len(raw)} bytes):")
if raw.strip():
    df = pd.read_csv(io.StringIO(raw), sep="\t")
    print(df.to_string(index=False))
else:
    print("(empty)")


dist inputs: ['cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz', 'cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz']
dist job: 5b623a3e-6ac7-41bb-9de4-c6f717cc0225


  state=complete  exit=[0]  waited=40s

dist.tsv (84 bytes):
Empty DataFrame
Columns: [Ref_file, Query_file, ANI, Align_fraction_ref, Align_fraction_query, Ref_name, Query_name]
Index: []


## 5. End-to-end check

In [5]:
checks = {
    "triangle job complete (exit 0)": triangle_job.get_job_status()["state"] == "complete"
                                          and triangle_job.get_exit_codes().get("exit_codes") == [0],
    "dist job complete (exit 0)":     dist_job.get_job_status()["state"] == "complete"
                                          and dist_job.get_exit_codes().get("exit_codes") == [0],
    "triangle TSV exists":            len(tri_outs) == 1,
    "dist TSV exists":                len(dist_outs) == 1,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off; inspect above.")


  [x] triangle job complete (exit 0)
  [x] dist job complete (exit 0)
  [x] triangle TSV exists
  [x] dist TSV exists

All green.
